<a href="https://colab.research.google.com/github/owenjiao0129-png/pythonAI/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
get_ipython().system('pip -q install ipyleaflet')
from google.colab import output
output.enable_custom_widget_manager()
import requests
import pandas as pd
URL = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"
def fetch_youbike():
    d = pd.DataFrame(requests.get(URL, timeout=10).json())
    rename = {}
    if "latitude" in d.columns:  rename["latitude"] = "lat"
    if "longitude" in d.columns: rename["longitude"] = "lng"
    for c in ["Quantity", "tot", "total", "totalstationcount"]:
        if c in d.columns:
            rename[c] = "capacity"
            break
    d = d.rename(columns=rename)
    for col in ["available_rent_bikes", "available_return_bikes", "capacity", "lat", "lng"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    d = d.dropna(subset=["lat", "lng"]).reset_index(drop=True)
    return d
df = fetch_youbike()
print("總站數：", len(df))

總站數： 1774


In [3]:
from math import radians, sin, cos, sqrt, atan2
WALK_KMH = 4.8
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))
def nearest(lat, lng, mode="借車", top=5):
    if mode == "借車":
        kind = "available_rent_bikes"
    else:
        kind = "available_return_bikes"
    result = []
    for i, row in df.iterrows():
        if row[kind] <= 0:
            continue
        dist = haversine(lat, lng, row["lat"], row["lng"])
        walk = dist / WALK_KMH * 60
        result.append({
            "sna": row["sna"],
            "sarea": row["sarea"],
            "avail": row[kind],
            "dist": dist,
            "walk": walk,
            "lat": row["lat"],
            "lng": row["lng"]
        })
    result.sort(key=lambda x: x["dist"])
    return pd.DataFrame(result[:top])

In [4]:
from ipyleaflet import Map, Marker, CircleMarker, Polyline, AwesomeIcon
import ipywidgets as widgets
from IPython.display import display
m = Map(center=(25.0478, 121.5319), zoom=12, scroll_wheel_zoom=True)
for i, row in df.iterrows():
    station = CircleMarker(
        location=(row["lat"], row["lng"]),
        radius=3,
        color="blue",
        fill_color="blue",
        fill_opacity=0.5
    )
    m.add_layer(station)

mode_toggle = widgets.ToggleButtons(options=["借車", "還車"], value="借車", description="模式:")
clear_btn = widgets.Button(description="清除", button_style="warning", icon="eraser")
info_box = widgets.HTML(value="👉 滑鼠移到站點看車數；點地圖任一位置找最近 5 站")

dynamic_layers = []

In [5]:
def clear_dynamic(_=None):
    for layer in dynamic_layers:
        try:
            m.remove_layer(layer)
        except Exception:
            pass
    dynamic_layers.clear()
    info_box.value = "已清除"
def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lng = kwargs["coordinates"]
    clear_dynamic()

    mode = mode_toggle.value
    result = nearest(lat, lng, mode=mode, top=5)
    kind = "可借" if mode == "借車" else "可還"

    click_marker = Marker(location=(lat, lng),
                          icon=AwesomeIcon(name="user", marker_color="red"))
    m.add_layer(click_marker); dynamic_layers.append(click_marker)

    rows = []
    for i, row in result.iterrows():
        target = CircleMarker(
            location=(row["lat"], row["lng"]),
            radius=6,
            color="green",
            fill_color="green",
            fill_opacity=0.8
        )
        m.add_layer(target)
        dynamic_layers.append(target)
        line = Polyline(
            locations=[(lat, lng), (row["lat"], row["lng"])],
            color="green",
            weight=2
        )
        m.add_layer(line)
        dynamic_layers.append(line)
        info_text = f"<li>{row['sna']}：{kind} {int(row['avail'])} 輛 (距離 {row['dist']:.2f} 公里，走路約 {row['walk']:.1f} 分鐘)</li>"
        rows.append(info_text)
    info_box.value = (f"📍 最近、且{kind}的前 5 站（{mode}）"
                      + "".join(rows))

m.on_interaction(on_click)
clear_btn.on_click(clear_dynamic)

def on_mode_change(_):
    info_box.value = f"已切到「{mode_toggle.value}」，請重新點一下地圖"
mode_toggle.observe(on_mode_change, names="value")

display(widgets.HBox([mode_toggle, clear_btn]))
display(widgets.HBox([m, info_box]))